# 08 — Landmark-only robustness analysis

## Purpose

This notebook evaluates whether the main transcriptomic conclusions of the manuscript remain stable when analyses are restricted to LINCS L1000 landmark genes, i.e. directly measured genes rather than the full Level 5 gene space that includes inferred genes.

This analysis was added during manuscript revision in response to reviewer concerns regarding the use of LINCS Level 5 inferred transcriptomic profiles.

## Main questions

1. Are landmark genes available for the curated vitamin D perturbation dataset?
2. How many of the previously analyzed genes are directly measured landmark genes?
3. Do global transcriptional structure and major variance patterns remain comparable in the landmark-only space?
4. Are the top recurrent transcriptional responses and core-score behavior stable when restricted to landmark genes?
5. Which conclusions remain supported, weakened, or require qualification?

## Output policy

This notebook does not save tables or figures by default. Results are displayed in the notebook to keep revision-associated outputs minimal and avoid storing intermediate files.

To export selected tables or figures for the reviewer response or supplementary material, set `SAVE_OUTPUTS = True` in the setup cell.

---

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

SAVE_OUTPUTS = False

PROJECT_ROOT = Path.cwd()

# If the notebook is executed from notebooks/revision, move two levels up.
if PROJECT_ROOT.name == "revision":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
REVISION_DIR = RESULTS_DIR / "revision"
REVISION_TABLES_DIR = REVISION_DIR / "tables"
REVISION_FIGURES_DIR = REVISION_DIR / "figures"
REVISION_SUMMARIES_DIR = REVISION_DIR / "summaries"

for path in [
    REVISION_TABLES_DIR,
    REVISION_FIGURES_DIR,
    REVISION_SUMMARIES_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir:     {DATA_DIR}")
print(f"Results dir:  {RESULTS_DIR}")

In [ ]:
# Index only project-relevant data/results files.
# Avoid scanning .git, virtual environments, caches, and package files.

scan_roots = [
    DATA_DIR,
    RESULTS_DIR,
    PROJECT_ROOT / "docs" / "revision",
    PROJECT_ROOT / "notebooks" / "revision",
]

exclude_dirs = {
    ".git",
    "vitd_env",
    ".venv",
    "venv",
    "__pycache__",
    ".ipynb_checkpoints",
    ".pytest_cache",
}

files = []

for root in scan_roots:
    if not root.exists():
        continue

    for p in root.rglob("*"):
        if not p.is_file():
            continue

        rel = p.relative_to(PROJECT_ROOT)

        if any(part in exclude_dirs for part in rel.parts):
            continue

        files.append({
            "path": rel.as_posix(),
            "suffix": p.suffix.lower(),
            "size_mb": p.stat().st_size / 1024**2,
        })

file_index = (
    pd.DataFrame(files)
    .sort_values(["suffix", "path"])
    .reset_index(drop=True)
)

print(f"Indexed files: {len(file_index)}")
file_index.head(10)

In [ ]:
# Load LINCS gene metadata and identify landmark genes

geneinfo_path = DATA_DIR / "raw_data" / "geneinfo_beta.txt"
geneinfo = pd.read_csv(geneinfo_path, sep="\t")

print("geneinfo shape:", geneinfo.shape)
print("geneinfo columns:")
print(geneinfo.columns.tolist())

geneinfo.head()

In [ ]:
# Detect the landmark annotation column

possible_landmark_cols = [
    col for col in geneinfo.columns
    if any(token in col.lower() for token in ["landmark", "feature", "space", "lm"])
]

print("Possible landmark-related columns:")
print(possible_landmark_cols)

for col in possible_landmark_cols:
    print("\n" + "=" * 80)
    print(col)
    print(geneinfo[col].value_counts(dropna=False).head(30))

In [ ]:
# Load cleaned expression matrix and signature metadata

expression_path = DATA_DIR / "exports" / "expression_matrix_clean.parquet"
metadata_path = DATA_DIR / "exports" / "signature_metadata_with_core_score.csv"

expression = pd.read_parquet(expression_path)
metadata = pd.read_csv(metadata_path)

print("expression shape:", expression.shape)
print("expression first columns:")
print(expression.columns[:20].tolist())

print("\nmetadata shape:", metadata.shape)
print("metadata columns:")
print(metadata.columns.tolist())

display(expression.head())
display(metadata.head())

In [ ]:
# Build landmark-only expression matrix

# Ensure gene IDs have comparable types
geneinfo = geneinfo.copy()
geneinfo["gene_id"] = geneinfo["gene_id"].astype(str)

expression = expression.copy()
expression.index = expression.index.astype(str)
expression.index.name = "gene_id"

# Landmark genes are directly measured in LINCS L1000
landmark_gene_ids = set(
    geneinfo.loc[geneinfo["feature_space"].eq("landmark"), "gene_id"]
)

full_gene_ids = set(expression.index)

landmark_gene_ids_in_matrix = sorted(full_gene_ids.intersection(landmark_gene_ids))
non_landmark_gene_ids_in_matrix = sorted(full_gene_ids.difference(landmark_gene_ids))

expression_landmark = expression.loc[landmark_gene_ids_in_matrix].copy()

gene_space_summary = pd.DataFrame(
    [
        {
            "gene_space": "full_expression_matrix",
            "source": "expression_matrix_clean.parquet",
            "n_genes": expression.shape[0],
            "n_signatures": expression.shape[1],
            "note": "Full cleaned expression matrix used in the manuscript.",
        },
        {
            "gene_space": "landmark_only",
            "source": "expression_matrix_clean.parquet + geneinfo_beta.txt",
            "n_genes": expression_landmark.shape[0],
            "n_signatures": expression_landmark.shape[1],
            "note": "Subset restricted to directly measured LINCS landmark genes.",
        },
        {
            "gene_space": "non_landmark_in_matrix",
            "source": "expression_matrix_clean.parquet + geneinfo_beta.txt",
            "n_genes": len(non_landmark_gene_ids_in_matrix),
            "n_signatures": expression.shape[1],
            "note": "Genes in the expression matrix not annotated as landmark.",
        },
        {
            "gene_space": "landmark_genes_in_geneinfo",
            "source": "geneinfo_beta.txt",
            "n_genes": len(landmark_gene_ids),
            "n_signatures": "not_applicable",
            "note": "Reference count of LINCS landmark genes in gene metadata.",
        },
    ]
)

if SAVE_OUTPUTS:
    output_path = REVISION_TABLES_DIR / "landmark_only_gene_space_summary.csv"
    gene_space_summary.to_csv(output_path, index=False)
    print("Saved:", output_path.relative_to(PROJECT_ROOT))

gene_space_summary

In [ ]:
# Verify signature alignment between expression matrices and metadata

expression_sig_ids = pd.Index(expression.columns.astype(str))
landmark_sig_ids = pd.Index(expression_landmark.columns.astype(str))
metadata_sig_ids = pd.Index(metadata["sig_id"].astype(str))

alignment_summary = pd.DataFrame(
    [
        {
            "check": "full_expression_columns_equal_metadata_sig_id_order",
            "result": expression_sig_ids.equals(metadata_sig_ids),
        },
        {
            "check": "landmark_expression_columns_equal_metadata_sig_id_order",
            "result": landmark_sig_ids.equals(metadata_sig_ids),
        },
        {
            "check": "full_expression_signature_set_equals_metadata_set",
            "result": set(expression_sig_ids) == set(metadata_sig_ids),
        },
        {
            "check": "landmark_expression_signature_set_equals_metadata_set",
            "result": set(landmark_sig_ids) == set(metadata_sig_ids),
        },
    ]
)

missing_in_metadata = sorted(set(expression_sig_ids) - set(metadata_sig_ids))
missing_in_expression = sorted(set(metadata_sig_ids) - set(expression_sig_ids))

print("Missing in metadata:", len(missing_in_metadata))
print("Missing in expression:", len(missing_in_expression))

display(alignment_summary)

if missing_in_metadata:
    print("First missing in metadata:", missing_in_metadata[:10])

if missing_in_expression:
    print("First missing in expression:", missing_in_expression[:10])

In [ ]:
# Landmark-only PCA

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Expression matrix orientation:
# rows = genes, columns = signatures
# PCA requires rows = signatures, columns = genes
X_landmark = expression_landmark.T.copy()

# Safety checks
assert X_landmark.index.equals(pd.Index(metadata["sig_id"].astype(str)))
assert X_landmark.shape == (metadata.shape[0], expression_landmark.shape[0])

print("Landmark-only PCA input matrix:", X_landmark.shape)

# Standardize genes across signatures
X_landmark_scaled = StandardScaler().fit_transform(X_landmark)

# Compute PCA using all possible components
n_components = min(X_landmark_scaled.shape)
pca_landmark = PCA(n_components=n_components, random_state=42)
pca_scores = pca_landmark.fit_transform(X_landmark_scaled)

explained = pca_landmark.explained_variance_ratio_
cumulative = np.cumsum(explained)

def n_components_for_threshold(cumulative_variance, threshold):
    return int(np.searchsorted(cumulative_variance, threshold) + 1)

pca_variance_summary = pd.DataFrame(
    [
        {
            "gene_space": "landmark_only",
            "n_signatures": X_landmark.shape[0],
            "n_genes": X_landmark.shape[1],
            "pc1_variance": explained[0],
            "pc2_variance": explained[1],
            "pc1_pc2_cumulative_variance": cumulative[1],
            "n_pcs_50pct_variance": n_components_for_threshold(cumulative, 0.50),
            "n_pcs_80pct_variance": n_components_for_threshold(cumulative, 0.80),
            "n_pcs_90pct_variance": n_components_for_threshold(cumulative, 0.90),
        }
    ]
)

pca_coordinates_landmark = metadata.copy()
pca_coordinates_landmark["PC1"] = pca_scores[:, 0]
pca_coordinates_landmark["PC2"] = pca_scores[:, 1]

display(pca_variance_summary)
display(pca_coordinates_landmark[["sig_id", "cell_id", "pert_id", "pert_dose", "core_score", "PC1", "PC2"]].head())

In [ ]:
# Full-gene PCA computed with the same procedure for direct comparison

X_full = expression.T.copy()

# Safety checks
assert X_full.index.equals(pd.Index(metadata["sig_id"].astype(str)))
assert X_full.shape == (metadata.shape[0], expression.shape[0])

print("Full-gene PCA input matrix:", X_full.shape)

X_full_scaled = StandardScaler().fit_transform(X_full)

n_components_full = min(X_full_scaled.shape)
pca_full = PCA(n_components=n_components_full, random_state=42)
pca_scores_full = pca_full.fit_transform(X_full_scaled)

explained_full = pca_full.explained_variance_ratio_
cumulative_full = np.cumsum(explained_full)

pca_variance_summary_full = pd.DataFrame(
    [
        {
            "gene_space": "full_gene_space",
            "n_signatures": X_full.shape[0],
            "n_genes": X_full.shape[1],
            "pc1_variance": explained_full[0],
            "pc2_variance": explained_full[1],
            "pc1_pc2_cumulative_variance": cumulative_full[1],
            "n_pcs_50pct_variance": n_components_for_threshold(cumulative_full, 0.50),
            "n_pcs_80pct_variance": n_components_for_threshold(cumulative_full, 0.80),
            "n_pcs_90pct_variance": n_components_for_threshold(cumulative_full, 0.90),
        }
    ]
)

pca_coordinates_full = metadata.copy()
pca_coordinates_full["PC1"] = pca_scores_full[:, 0]
pca_coordinates_full["PC2"] = pca_scores_full[:, 1]

pca_variance_comparison = pd.concat(
    [pca_variance_summary_full, pca_variance_summary],
    ignore_index=True,
)

# Compare first two PC coordinate structures.
# Absolute correlations are used because PCA axes can flip sign.
pc_coordinate_correlation = pd.DataFrame(
    [
        {
            "comparison": "full_PC1_vs_landmark_PC1",
            "spearman_abs_r": abs(
                pd.Series(pca_scores_full[:, 0]).corr(
                    pd.Series(pca_scores[:, 0]),
                    method="spearman",
                )
            ),
            "pearson_abs_r": abs(
                pd.Series(pca_scores_full[:, 0]).corr(
                    pd.Series(pca_scores[:, 0]),
                    method="pearson",
                )
            ),
        },
        {
            "comparison": "full_PC2_vs_landmark_PC2",
            "spearman_abs_r": abs(
                pd.Series(pca_scores_full[:, 1]).corr(
                    pd.Series(pca_scores[:, 1]),
                    method="spearman",
                )
            ),
            "pearson_abs_r": abs(
                pd.Series(pca_scores_full[:, 1]).corr(
                    pd.Series(pca_scores[:, 1]),
                    method="pearson",
                )
            ),
        },
        {
            "comparison": "full_PC1_vs_landmark_PC2",
            "spearman_abs_r": abs(
                pd.Series(pca_scores_full[:, 0]).corr(
                    pd.Series(pca_scores[:, 1]),
                    method="spearman",
                )
            ),
            "pearson_abs_r": abs(
                pd.Series(pca_scores_full[:, 0]).corr(
                    pd.Series(pca_scores[:, 1]),
                    method="pearson",
                )
            ),
        },
        {
            "comparison": "full_PC2_vs_landmark_PC1",
            "spearman_abs_r": abs(
                pd.Series(pca_scores_full[:, 1]).corr(
                    pd.Series(pca_scores[:, 0]),
                    method="spearman",
                )
            ),
            "pearson_abs_r": abs(
                pd.Series(pca_scores_full[:, 1]).corr(
                    pd.Series(pca_scores[:, 0]),
                    method="pearson",
                )
            ),
        },
    ]
)

if SAVE_OUTPUTS:
    comparison_output = REVISION_TABLES_DIR / "landmark_vs_full_pca_variance_summary.csv"
    pca_variance_comparison.to_csv(comparison_output, index=False)
    print("Saved:", comparison_output.relative_to(PROJECT_ROOT))

display(pca_variance_comparison)
display(pc_coordinate_correlation)

In [ ]:
# Compare PCA subspaces between full-gene and landmark-only analyses.
# This is more appropriate than comparing PC1-to-PC1 directly, because PCA axes can rotate or swap.

def principal_angle_summary(scores_a, scores_b, n_dims_list=(2, 5, 10, 20, 50)):
    rows = []

    for n_dims in n_dims_list:
        n_dims = min(n_dims, scores_a.shape[1], scores_b.shape[1])

        A = scores_a[:, :n_dims]
        B = scores_b[:, :n_dims]

        # Center columns
        A = A - A.mean(axis=0)
        B = B - B.mean(axis=0)

        # Orthonormal bases for each score subspace
        QA, _ = np.linalg.qr(A)
        QB, _ = np.linalg.qr(B)

        # Singular values of QA.T @ QB are cosines of principal angles
        singular_values = np.linalg.svd(QA.T @ QB, compute_uv=False)
        singular_values = np.clip(singular_values, -1, 1)

        angles_degrees = np.degrees(np.arccos(singular_values))

        rows.append(
            {
                "n_pcs_compared": n_dims,
                "mean_canonical_correlation": float(np.mean(singular_values)),
                "min_canonical_correlation": float(np.min(singular_values)),
                "max_principal_angle_degrees": float(np.max(angles_degrees)),
                "mean_principal_angle_degrees": float(np.mean(angles_degrees)),
            }
        )

    return pd.DataFrame(rows)


pca_subspace_similarity = principal_angle_summary(
    scores_a=pca_scores_full,
    scores_b=pca_scores,
    n_dims_list=(2, 5, 10, 20, 50),
)

if SAVE_OUTPUTS:
    subspace_output = REVISION_TABLES_DIR / "landmark_vs_full_pca_subspace_similarity.csv"
    pca_subspace_similarity.to_csv(subspace_output, index=False)
    print("Saved:", subspace_output.relative_to(PROJECT_ROOT))

pca_subspace_similarity

In [ ]:
# Landmark-only PCA figure: two-panel layout
# Panel A: colored by cell line
# Panel B: colored by compound / perturbagen label

import matplotlib.pyplot as plt

compound_meta_path = DATA_DIR / "exports" / "subset_compounds_meta.csv"

pca_plot_data = pca_coordinates_landmark.copy()

# Add readable compound labels if compound metadata is available
if compound_meta_path.exists():
    compound_meta = pd.read_csv(compound_meta_path)
    print("compound_meta columns:", compound_meta.columns.tolist())

    possible_name_cols = [
        col for col in compound_meta.columns
        if col.lower() in ["cmap_name", "pert_iname", "compound_name", "name"]
        or "name" in col.lower()
    ]

    if "pert_id" in compound_meta.columns and possible_name_cols:
        compound_name_col = possible_name_cols[0]
        pca_plot_data = pca_plot_data.merge(
            compound_meta[["pert_id", compound_name_col]].drop_duplicates(),
            on="pert_id",
            how="left",
        )
        pca_plot_data["compound_label"] = pca_plot_data[compound_name_col].fillna(
            pca_plot_data["pert_id"]
        )
        print(f"Using compound label column: {compound_name_col}")
    else:
        pca_plot_data["compound_label"] = pca_plot_data["pert_id"]
        print("No compound-name column detected; using pert_id.")
else:
    pca_plot_data["compound_label"] = pca_plot_data["pert_id"]
    print("Compound metadata not found; using pert_id.")

# Create combined figure
fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(13, 5),
    sharex=True,
    sharey=True,
)

# Panel A — cell line
ax = axes[0]

for label, group in pca_plot_data.groupby("cell_id"):
    ax.scatter(
        group["PC1"],
        group["PC2"],
        label=label,
        alpha=0.8,
        s=35,
    )

ax.set_title("A. Landmark-only PCA by cell line")
ax.set_xlabel(f"PC1 ({explained[0] * 100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({explained[1] * 100:.1f}% variance)")
ax.grid(True, linewidth=0.3, alpha=0.5)
ax.legend(title="Cell line", fontsize=8, title_fontsize=9)

# Panel B — compound
ax = axes[1]

for label, group in pca_plot_data.groupby("compound_label"):
    ax.scatter(
        group["PC1"],
        group["PC2"],
        label=label,
        alpha=0.8,
        s=35,
    )

ax.set_title("B. Landmark-only PCA by compound")
ax.set_xlabel(f"PC1 ({explained[0] * 100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({explained[1] * 100:.1f}% variance)")
ax.grid(True, linewidth=0.3, alpha=0.5)
ax.legend(title="Compound", fontsize=8, title_fontsize=9)

fig.suptitle(
    "Landmark-only PCA of vitamin D-related LINCS signatures",
    y=1.03,
)

fig.tight_layout()

if SAVE_OUTPUTS:
    combined_fig_path = REVISION_FIGURES_DIR / "landmark_only_pca_two_panel.svg"
    fig.savefig(combined_fig_path, bbox_inches="tight")
    print("Saved:", combined_fig_path.relative_to(PROJECT_ROOT))

plt.show()

In [ ]:
# Landmark-only metadata association with PCA structure
# This provides a lightweight robustness check of whether cell line, compound, and dose
# remain associated with the dominant landmark-only transcriptional structure.

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder

def adjusted_r2_score(y, X):
    """
    Compute adjusted R2 for multivariate PCA coordinates using total sum of squares.
    y: array-like, shape (n_samples, n_outputs)
    X: array-like, shape (n_samples, n_predictors)
    """
    y = np.asarray(y)
    X = np.asarray(X)

    model = LinearRegression()
    model.fit(X, y)
    y_pred = model.predict(X)

    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - y.mean(axis=0)) ** 2)

    r2 = 1 - (ss_res / ss_tot)

    n = y.shape[0]
    p = X.shape[1]

    if n <= p + 1:
        adj_r2 = np.nan
    else:
        adj_r2 = 1 - (1 - r2) * ((n - 1) / (n - p - 1))

    return r2, adj_r2


# Use the same number of PCs needed to explain ~50% variance in landmark-only space.
n_pcs_for_model = int(pca_variance_summary.loc[0, "n_pcs_50pct_variance"])
Y_landmark_pcs = pca_scores[:, :n_pcs_for_model]

model_data = metadata.copy()
model_data["compound_label"] = pca_plot_data["compound_label"].values

# Categorical encoders
encoder_cell = OneHotEncoder(drop="first", sparse_output=False)
X_cell = encoder_cell.fit_transform(model_data[["cell_id"]])

encoder_compound = OneHotEncoder(drop="first", sparse_output=False)
X_compound = encoder_compound.fit_transform(model_data[["compound_label"]])

encoder_dose_bin = OneHotEncoder(drop="first", sparse_output=False)
X_dose_bin = encoder_dose_bin.fit_transform(model_data[["dose_bin"]])

# Continuous dose, log10-transformed
X_log_dose = np.log10(model_data[["pert_dose"]].astype(float).values)

rows = []

for factor_name, X in [
    ("cell_id", X_cell),
    ("compound_label", X_compound),
    ("dose_bin", X_dose_bin),
    ("log10_pert_dose", X_log_dose),
]:
    r2, adj_r2 = adjusted_r2_score(Y_landmark_pcs, X)
    rows.append(
        {
            "gene_space": "landmark_only",
            "pca_space": f"first_{n_pcs_for_model}_PCs_approx_50pct_variance",
            "factor": factor_name,
            "n_predictors": X.shape[1],
            "r2": r2,
            "adjusted_r2": adj_r2,
        }
    )

landmark_pca_factor_association = (
    pd.DataFrame(rows)
    .sort_values("r2", ascending=False)
    .reset_index(drop=True)
)

if SAVE_OUTPUTS:
    factor_output = REVISION_TABLES_DIR / "landmark_only_pca_factor_association.csv"
    landmark_pca_factor_association.to_csv(factor_output, index=False)
    print("Saved:", factor_output.relative_to(PROJECT_ROOT))

landmark_pca_factor_association

---

## Landmark-only consensus core and core_score robustness

In [ ]:
# Annotate expression genes with LINCS gene metadata

gene_annotation = (
    geneinfo[["gene_id", "gene_symbol", "gene_title", "gene_type", "feature_space"]]
    .copy()
)

gene_annotation["gene_id"] = gene_annotation["gene_id"].astype(str)

# Mean expression/effect by cell line in the landmark-only matrix
effects_by_cell_landmark = {}

for cell in metadata["cell_id"].unique():
    sig_ids = metadata.loc[metadata["cell_id"].eq(cell), "sig_id"].astype(str)
    effects_by_cell_landmark[cell] = expression_landmark[sig_ids].mean(axis=1)

effects_by_cell_landmark = pd.DataFrame(effects_by_cell_landmark)
effects_by_cell_landmark.index.name = "gene_id"

effects_by_cell_landmark_annotated = (
    effects_by_cell_landmark
    .reset_index()
    .merge(gene_annotation, on="gene_id", how="left")
)

# Reorder columns for readability
front_cols = ["gene_id", "gene_symbol", "gene_title", "gene_type", "feature_space"]
cell_cols = [col for col in effects_by_cell_landmark_annotated.columns if col not in front_cols]
effects_by_cell_landmark_annotated = effects_by_cell_landmark_annotated[front_cols + cell_cols]

print("Landmark effects_by_cell:", effects_by_cell_landmark.shape)

effects_by_cell_landmark_annotated.head()

In [ ]:
# Re-derive landmark-only consensus gene sets using the same vote-count strategy
# used in the manuscript core_score definition.

TOP_N = 50
MIN_VOTES = 2

landmark_up_votes = {}
landmark_down_votes = {}

for cell in effects_by_cell_landmark.columns:
    cell_effects = effects_by_cell_landmark[cell].dropna()

    top_up = cell_effects.sort_values(ascending=False).head(TOP_N).index.astype(str)
    top_down = cell_effects.sort_values(ascending=True).head(TOP_N).index.astype(str)

    for gene_id in top_up:
        landmark_up_votes.setdefault(gene_id, []).append(cell)

    for gene_id in top_down:
        landmark_down_votes.setdefault(gene_id, []).append(cell)

landmark_up_core_ids = sorted(
    [gene_id for gene_id, cells in landmark_up_votes.items() if len(cells) >= MIN_VOTES]
)

landmark_down_core_ids = sorted(
    [gene_id for gene_id, cells in landmark_down_votes.items() if len(cells) >= MIN_VOTES]
)

landmark_core_ids = sorted(set(landmark_up_core_ids) | set(landmark_down_core_ids))

print("Landmark-only up core genes:", len(landmark_up_core_ids))
print("Landmark-only down core genes:", len(landmark_down_core_ids))
print("Landmark-only total unique core genes:", len(landmark_core_ids))

# Build detailed vote table
rows = []

for gene_id in landmark_core_ids:
    up_cells = landmark_up_votes.get(gene_id, [])
    down_cells = landmark_down_votes.get(gene_id, [])

    rows.append(
        {
            "gene_id": gene_id,
            "up_votes": len(up_cells),
            "down_votes": len(down_cells),
            "up_cells": ";".join(up_cells),
            "down_cells": ";".join(down_cells),
            "landmark_core_direction": (
                "up" if gene_id in landmark_up_core_ids and gene_id not in landmark_down_core_ids
                else "down" if gene_id in landmark_down_core_ids and gene_id not in landmark_up_core_ids
                else "discordant"
            ),
        }
    )

landmark_core_vote_table = (
    pd.DataFrame(rows)
    .merge(gene_annotation, on="gene_id", how="left")
)

# Add mean effects by cell line
landmark_core_vote_table = landmark_core_vote_table.merge(
    effects_by_cell_landmark.reset_index(),
    on="gene_id",
    how="left",
)

# Reorder
front_cols = [
    "gene_id",
    "gene_symbol",
    "gene_title",
    "gene_type",
    "feature_space",
    "landmark_core_direction",
    "up_votes",
    "down_votes",
    "up_cells",
    "down_cells",
]
other_cols = [c for c in landmark_core_vote_table.columns if c not in front_cols]
landmark_core_vote_table = landmark_core_vote_table[front_cols + other_cols]

if SAVE_OUTPUTS:
    landmark_core_output = REVISION_TABLES_DIR / "landmark_only_consensus_core_vote_table.csv"
    landmark_core_vote_table.to_csv(landmark_core_output, index=False)
    print("Saved:", landmark_core_output.relative_to(PROJECT_ROOT))

landmark_core_vote_table.sort_values(
    ["landmark_core_direction", "up_votes", "down_votes", "gene_symbol"],
    ascending=[True, False, False, True],
).head(20)

In [ ]:
# Compute landmark-only core_score and compare it with the original full-space core_score

landmark_up_core_ids = [gene_id for gene_id in landmark_up_core_ids if gene_id in expression_landmark.index]
landmark_down_core_ids = [gene_id for gene_id in landmark_down_core_ids if gene_id in expression_landmark.index]

landmark_core_score = (
    expression_landmark.loc[landmark_up_core_ids].mean(axis=0)
    - expression_landmark.loc[landmark_down_core_ids].mean(axis=0)
)

landmark_core_score = landmark_core_score.rename("landmark_core_score")

core_score_comparison = metadata[["sig_id", "cell_id", "pert_id", "pert_dose", "dose_bin", "core_score"]].copy()
core_score_comparison["landmark_core_score"] = core_score_comparison["sig_id"].map(landmark_core_score)

# Sanity check
missing_landmark_scores = core_score_comparison["landmark_core_score"].isna().sum()
print("Missing landmark core scores:", missing_landmark_scores)

# Global correlations
global_core_score_correlation = pd.DataFrame(
    [
        {
            "comparison": "original_core_score_vs_landmark_core_score",
            "n_signatures": core_score_comparison.shape[0],
            "spearman_r": core_score_comparison["core_score"].corr(
                core_score_comparison["landmark_core_score"],
                method="spearman",
            ),
            "pearson_r": core_score_comparison["core_score"].corr(
                core_score_comparison["landmark_core_score"],
                method="pearson",
            ),
        }
    ]
)

# Correlations by cell line
cell_core_score_correlation = []

for cell, group in core_score_comparison.groupby("cell_id"):
    cell_core_score_correlation.append(
        {
            "cell_id": cell,
            "n_signatures": group.shape[0],
            "spearman_r": group["core_score"].corr(
                group["landmark_core_score"],
                method="spearman",
            ),
            "pearson_r": group["core_score"].corr(
                group["landmark_core_score"],
                method="pearson",
            ),
            "original_core_score_mean": group["core_score"].mean(),
            "landmark_core_score_mean": group["landmark_core_score"].mean(),
            "original_core_score_sd": group["core_score"].std(),
            "landmark_core_score_sd": group["landmark_core_score"].std(),
        }
    )

cell_core_score_correlation = pd.DataFrame(cell_core_score_correlation)

display(global_core_score_correlation)
display(cell_core_score_correlation)
display(core_score_comparison.head())

In [ ]:
# Standardized comparison between original core_score and landmark-only core_score

from scipy.stats import zscore
import matplotlib.pyplot as plt

core_score_comparison = core_score_comparison.copy()

# Global z-scores
core_score_comparison["core_score_z"] = zscore(core_score_comparison["core_score"])
core_score_comparison["landmark_core_score_z"] = zscore(core_score_comparison["landmark_core_score"])

# Cell-line-centered z-scores
core_score_comparison["core_score_z_by_cell"] = (
    core_score_comparison
    .groupby("cell_id")["core_score"]
    .transform(lambda x: zscore(x, ddof=0))
)

core_score_comparison["landmark_core_score_z_by_cell"] = (
    core_score_comparison
    .groupby("cell_id")["landmark_core_score"]
    .transform(lambda x: zscore(x, ddof=0))
)

standardized_core_score_correlation = pd.DataFrame(
    [
        {
            "comparison": "global_z_scores",
            "n_signatures": core_score_comparison.shape[0],
            "spearman_r": core_score_comparison["core_score_z"].corr(
                core_score_comparison["landmark_core_score_z"],
                method="spearman",
            ),
            "pearson_r": core_score_comparison["core_score_z"].corr(
                core_score_comparison["landmark_core_score_z"],
                method="pearson",
            ),
        },
        {
            "comparison": "cell_line_centered_z_scores",
            "n_signatures": core_score_comparison.shape[0],
            "spearman_r": core_score_comparison["core_score_z_by_cell"].corr(
                core_score_comparison["landmark_core_score_z_by_cell"],
                method="spearman",
            ),
            "pearson_r": core_score_comparison["core_score_z_by_cell"].corr(
                core_score_comparison["landmark_core_score_z_by_cell"],
                method="pearson",
            ),
        },
    ]
)

# Plot standardized score comparison
fig, ax = plt.subplots(figsize=(6, 5))

for cell, group in core_score_comparison.groupby("cell_id"):
    ax.scatter(
        group["core_score_z"],
        group["landmark_core_score_z"],
        label=cell,
        alpha=0.8,
        s=35,
    )

ax.axhline(0, linewidth=0.8)
ax.axvline(0, linewidth=0.8)
ax.set_xlabel("Original core_score (global z-score)")
ax.set_ylabel("Landmark-only core_score (global z-score)")
ax.set_title("Concordance between original and landmark-only core activity scores")
ax.legend(title="Cell line", fontsize=8, title_fontsize=9)
ax.grid(True, linewidth=0.3, alpha=0.5)

fig.tight_layout()

if SAVE_OUTPUTS:
    core_score_fig_path = REVISION_FIGURES_DIR / "landmark_vs_original_core_score_scatter.svg"
    fig.savefig(core_score_fig_path, bbox_inches="tight")
    print("Saved:", core_score_fig_path.relative_to(PROJECT_ROOT))

plt.show()

standardized_core_score_correlation

In [ ]:
# Dose-score association using landmark-only core_score

from scipy.stats import spearmanr
import statsmodels.api as sm

dose_rows = []

for cell, group in core_score_comparison.groupby("cell_id"):
    group = group.copy()
    group = group.dropna(subset=["pert_dose", "landmark_core_score"])
    group = group[group["pert_dose"] > 0]

    group["log10_dose"] = np.log10(group["pert_dose"].astype(float))

    # Spearman monotonic association
    rho, spearman_p = spearmanr(group["log10_dose"], group["landmark_core_score"])

    # Linear model with HC3 robust SE
    X = sm.add_constant(group["log10_dose"])
    y = group["landmark_core_score"]

    model = sm.OLS(y, X).fit(cov_type="HC3")

    slope = model.params["log10_dose"]
    slope_ci_low, slope_ci_high = model.conf_int().loc["log10_dose"].tolist()
    slope_p = model.pvalues["log10_dose"]

    dose_rows.append(
        {
            "gene_space": "landmark_only",
            "cell_id": cell,
            "n_signatures": group.shape[0],
            "spearman_rho": rho,
            "spearman_p": spearman_p,
            "ols_slope_log10_dose": slope,
            "ols_slope_ci_low": slope_ci_low,
            "ols_slope_ci_high": slope_ci_high,
            "ols_slope_p_hc3": slope_p,
        }
    )

landmark_dose_response_summary = pd.DataFrame(dose_rows)

landmark_dose_response_summary

In [ ]:
# Compare original and landmark-only dose-score associations using the same model specification

def compute_dose_response_summary(score_column, gene_space_label):
    rows = []

    for cell, group in core_score_comparison.groupby("cell_id"):
        group = group.copy()
        group = group.dropna(subset=["pert_dose", score_column])
        group = group[group["pert_dose"] > 0]

        group["log10_dose"] = np.log10(group["pert_dose"].astype(float))

        rho, spearman_p = spearmanr(group["log10_dose"], group[score_column])

        X = sm.add_constant(group["log10_dose"])
        y = group[score_column]

        model = sm.OLS(y, X).fit(cov_type="HC3")

        slope = model.params["log10_dose"]
        slope_ci_low, slope_ci_high = model.conf_int().loc["log10_dose"].tolist()
        slope_p = model.pvalues["log10_dose"]

        rows.append(
            {
                "gene_space": gene_space_label,
                "score_column": score_column,
                "cell_id": cell,
                "n_signatures": group.shape[0],
                "spearman_rho": rho,
                "spearman_p": spearman_p,
                "ols_slope_log10_dose": slope,
                "ols_slope_ci_low": slope_ci_low,
                "ols_slope_ci_high": slope_ci_high,
                "ols_slope_p_hc3": slope_p,
            }
        )

    return pd.DataFrame(rows)


original_dose_response_summary = compute_dose_response_summary(
    score_column="core_score",
    gene_space_label="full_gene_space_original_core_score",
)

landmark_dose_response_summary = compute_dose_response_summary(
    score_column="landmark_core_score",
    gene_space_label="landmark_only_core_score",
)

dose_response_comparison = pd.concat(
    [original_dose_response_summary, landmark_dose_response_summary],
    ignore_index=True,
)

dose_response_comparison = dose_response_comparison.sort_values(
    ["cell_id", "gene_space"]
).reset_index(drop=True)

if SAVE_OUTPUTS:
    dose_comparison_output = REVISION_TABLES_DIR / "landmark_vs_original_dose_response_comparison.csv"
    dose_response_comparison.to_csv(dose_comparison_output, index=False)
    print("Saved:", dose_comparison_output.relative_to(PROJECT_ROOT))

dose_response_comparison

## Summary

The 978 directly measured LINCS landmark genes were fully recovered in the cleaned expression matrix, allowing a landmark-only robustness analysis without reprocessing the raw GCTX file.

The landmark-only analysis preserved the main patterns observed in the full Level 5 gene space. Global PCA structure remained comparable, although the axes were rotated/interchanged, and cell line identity remained the strongest metadata-associated factor relative to compound identity or dose.

A landmark-only consensus core was reconstructed using the same vote-count strategy as in the manuscript. The resulting landmark-only core_score was highly concordant with the original core_score across signatures, both globally and within cell lines.

Dose-score trends were also broadly preserved, particularly in A549, MCF7 and PC3. HA1E remained weak/non-significant, while U2OS showed greater uncertainty, consistent with its smaller number of signatures.

Overall, these results support that the main conclusions are not driven solely by LINCS inferred genes, while reinforcing the need to interpret dose-related patterns as dataset-level transcriptional trends.
